In [33]:
import ccxt
import pandas as pd
import numpy as np
import vectorbt as vbt
import matplotlib.pyplot as plt

data = pd.read_csv(
    'BTCUSDT_5m.csv',
    parse_dates=['Timestamp'],
    index_col='Timestamp'
)
data = data.sort_index()

data['Session'] = data.index.date
data['Time'] = data.index.time
# --------------------------------
# 2. Create VWAP-like Anchor
# --------------------------------
_open = data['Open']
high = data['High']
low = data['Low']
close = data['Close']
volume = data['Volume']

typical_price = (high + low + close) / 3

data['TPV'] = typical_price * volume

data['Cum_TPV'] = (data.groupby('Session')['TPV'].cumsum())

data['Cum_Volume'] = (data.groupby('Session')['Volume'].cumsum())
data['VWAP'] = (data['Cum_TPV'] / data['Cum_Volume'])

# --------------------------------
# 3. Create Signals
# --------------------------------
z_thresholds = np.arange(
    -1,
    -4,
    -0.25
)
#thresholds = -2
stop_losses = 0.01

#data['Deviation'] = ((data['Close'] - data['VWAP']) / data['VWAP']) * 100
#Calculate ATR
tr1 = high - low
tr2 = abs(high - close.shift(1))
tr3 = abs(low - close.shift(1))

true_range = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

data['ATR'] = (true_range.rolling(50).mean())
data['ATR_Pct'] = (data['ATR'] / close) * 100
data['volty_filter'] = (data['ATR_Pct'] < 0.15)


data['Rolling_STD'] = (data['Close'].rolling(20).std())
data['ZScore'] = (data['Close'] - data['VWAP']) / data['Rolling_STD']

data['Session_Open'] = (data.groupby('Session')['Open'].transform('first'))
data['Move_From_Open'] = ((data['Close'] - data['Session_Open']) / data['Session_Open']) * 100
data['trend_filter'] = (data['Move_From_Open'].abs() < 1.0)

#Time Filter
data['Hour'] = data.index.hour
data['time_filter'] = ((data['Hour'] >= 13) & (data['Hour'] <= 20))

#Exit
#data['Long_Exit'] = (data['Close'].shift(1) <= data['VWAP'].shift(1)) & (data['Close'] > data['VWAP'])
data['Long_Exit'] = data['ZScore'] <= 0 
#---------------------------------
#EOD Exit at 1525
eod_exit = (data['Time'].astype(str) == '15:25:00')
#---------------------------------
data['Exit'] = (data['Long_Exit'] | eod_exit)

#Walk Forward
train_data = data.loc[
    '2024-01-01':'2025-12-31'
]
test_data = data.loc[
    '2026-01-01':'2026-05-12'
]
train_entries = pd.DataFrame({z: (
        (
            train_data['ZScore'].shift(1) < z
        )
        &
        (
            train_data['ZScore'] >= z
        )
        &
        train_data['trend_filter']
        &
        train_data['volty_filter']
        &
        train_data['time_filter']
    )
    for z in z_thresholds
})

train_exits = pd.DataFrame({
    z: train_data['Exit']
    for z in z_thresholds
})
print(train_entries.shape)
print(train_exits.shape)
# -------------------------------
# 4. Portfolio Backtest
# --------------------------------
train_portfolio = vbt.Portfolio.from_signals(
    close = train_data['Close'],
    entries = train_entries,
    exits = train_exits,
    fees=0,
    sl_stop=0.02,    
    freq='5min',
    init_cash=100000,
    #cash_sharing=True
 #   broadcast_kwargs=dict(columns_from='keep')
)
train_return = train_portfolio.total_return()

best_z = train_return.idxmax()

print(best_z)
# --------------------------------
# 5. Stats
# --------------------------------
print(train_portfolio.stats())

'''results = pd.DataFrame({
    'Total_Return': portfolio.total_return(),
    'Sharpe': portfolio.sharpe_ratio(),
    'Max_Drawdown': portfolio.max_drawdown(),
    'Trades': portfolio.trades.count()
})'''

#print(results)
# --------------------------------
# 6. Plot
# --------------------------------
#portfolio.value().vbt.plot().show()


(210528, 12)
(210528, 12)
-1.0
Start                               2024-01-01 00:00:00
End                                 2025-12-31 23:55:00
Period                                731 days 00:00:00
Start Value                                    100000.0
End Value                                  99246.248613
Total Return [%]                              -0.753751
Benchmark Return [%]                         106.731006
Max Gross Exposure [%]                            100.0
Total Fees Paid                                     0.0
Max Drawdown [%]                               1.718161
Max Drawdown Duration                 643 days 10:56:15
Total Trades                                  27.916667
Total Closed Trades                           27.916667
Total Open Trades                                   0.0
Open Trade PnL                                      0.0
Win Rate [%]                                  15.549315
Best Trade [%]                                 1.054317
Worst Trade [%]  

C:\Users\Mani\AppData\Local\Temp\ipykernel_15860\1579124217.py:135: UserWarning: Object has multiple columns. Aggregating using <function mean at 0x000002069A185120>. Pass column to select a single column/group.
  print(train_portfolio.stats())


"results = pd.DataFrame({\n    'Total_Return': portfolio.total_return(),\n    'Sharpe': portfolio.sharpe_ratio(),\n    'Max_Drawdown': portfolio.max_drawdown(),\n    'Trades': portfolio.trades.count()\n})"